# CTI K/V resolution: K2 vs K4 (SPFI ×4, EMPS, Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/Tehila_Navon/notebooks/cti_scales_colab.ipynb)

**Question.** In the CTI block (Cross-token Integration, `algo/cti.py`), how small can the K and V
maps be before SPFI's super-resolution quality drops, and how much computation does it save?

**How CTI builds K and V** (for a 64×64 feature map):
1. Three branches, each a depth-wise separable conv: 3×3 / stride 1, 5×5 / stride 2, 7×7 / stride 4.
   Depth-wise: one filter per channel, looks at neighbours without mixing channels.
   Point-wise (1×1): mixes the channels at each position without looking at neighbours.
2. The branches come out at 64×64, 32×32 and 16×16. This is done separately for K and for V (6 maps).
3. To combine them, all branches are resized (bilinear) to one common size, mixed, and fused by a 1×1 conv.

**What changes: K, the common size in step 3.**

| Config | K and V size | K/V tokens per attention | Q tokens |
|---|---|---|---|
| **K2** | 32×32 (H/2) | 1,024 | 4,096 |
| **K4** | 16×16 (H/4) | 256 | 4,096 |

Q keeps the full 64×64 map, so the output keeps full resolution. Smaller K/V means each of the 4,096
queries compares against fewer positions, which is where the computation is saved.

**What stays the same.** Everything else: the three branches, the SPFI model (`algo/model.py`, `algo/sci.py`), loss
(Charbonnier + 0.5·(1−SSIM)), optimizer (AdamW, lr 1e-4), ×4 scale, 64×64 LR patches, the seed,
the initial weights, and the exact order of training patches.

**Protocol.**
* Data: only EMPS's `train.csv` images (366), split 80/20 into training / validation, the same as
  `train_colab.ipynb`. The 99 `test.csv` images are never used.
* Each config trains for the same number of iterations (batch size 1).
* Every `EVAL_EVERY` iterations: PSNR and SSIM on 32 fixed validation patches. The table reports the
  best check and the last check, since the last step alone is noisy.
* Cost: parameters, forward GFLOPs on a 64×64 LR input, and GPU forward time.
* Baseline: plain bicubic upscaling on the same validation patches. A useful model must beat it.

The experiment code is `algo/compare_cti_scales.py`; this notebook only runs it and shows the results.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")

## Settings

In [ ]:
from pathlib import Path

CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "Tehila_Navon"

EMPS_URL = "https://github.com/by256/emps.git"
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"   # same pinned release as train_colab.ipynb

CONFIGS = ["K2", "K4"]     # to run one at a time (e.g. after a disconnect): ["K4"]; earlier results in OUT_DIR are kept
ITERS = 5000               # training iterations per config
EVAL_EVERY = 500           # PSNR / SSIM check interval
SEED = 0

SAVE_TO_DRIVE = True       # keep results.json + log in Google Drive in case Colab disconnects

CODE_DIR = Path("/content/SR_amat_3")
EMPS_DIR = Path("/content/emps")
OUT_DIR = Path("/content/cti_scales_compare")

## Code, data and dependencies

In [ ]:
!pip install -q "pytorch_wavelets>=1.3" "PyWavelets>=1.6" "setuptools<81"

import subprocess

def run(*cmd, cwd=None):
    subprocess.run(list(cmd), cwd=cwd, check=True)

if not CODE_DIR.exists():
    run("git", "clone", "-q", "-b", CODE_BRANCH, CODE_REPO_URL, str(CODE_DIR))
else:
    run("git", "pull", "-q", cwd=CODE_DIR)
print("code:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=CODE_DIR,
                              capture_output=True, text=True).stdout.strip())

if not EMPS_DIR.exists():
    run("git", "clone", "-q", EMPS_URL, str(EMPS_DIR))
run("git", "checkout", "-q", EMPS_COMMIT, cwd=EMPS_DIR)
print("EMPS:", EMPS_COMMIT[:10], "|", len(list((EMPS_DIR / "images").glob("*.png"))), "images,",
      sum(1 for l in (EMPS_DIR / "train.csv").read_text().splitlines() if l.strip()), "in train.csv")

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/cti_scales_compare")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("results go to", OUT_DIR)

## Run the comparison
About 30 minutes per config on a T4. Progress is printed every 25 iterations and a PSNR / SSIM
check every `EVAL_EVERY`. The full log is also written to `OUT_DIR/log.txt`.

In [ ]:
%cd /content/SR_amat_3
configs = " ".join(f'"{c}"' for c in CONFIGS)
# -W ignore hides the harmless pkg_resources warning; -u + tee stream every line as it is printed.
!python -u -W ignore -m algo.compare_cti_scales --emps_dir {EMPS_DIR} --iters {ITERS} --eval_every {EVAL_EVERY} \
    --seed {SEED} --configs {configs} --out_dir {OUT_DIR} 2>&1 | tee {OUT_DIR}/log.txt

## Results

In [ ]:
import json
import pandas as pd

res = json.loads((OUT_DIR / "results.json").read_text())
R = res["results"]
bicubic = R["bicubic"]["psnr"]
NAMES = sorted(k for k in R if k != "bicubic")   # every config saved in results.json

rows = []
for name in NAMES:
    r = R[name]
    rows.append({
        "config": name,
        "K/V size": r["kv_size"],
        "params": f"{r['params']:,}",
        "GFLOPs": round(r["gflops"], 2),
        "fwd ms": round(r["latency_ms"]),
        "best PSNR (dB)": round(r["psnr"], 3),
        "best SSIM": round(r["ssim"], 4),
        "best @ iter": r["best_iter"],
        "last PSNR (dB)": round(r["psnr_last"], 3),
        "last SSIM": round(r["ssim_last"], 4),
        "best PSNR - bicubic": round(r["psnr"] - bicubic, 3),
    })
table = pd.DataFrame(rows).set_index("config")
print(f"iters={res['iters']}  seed={res['seed']}  val patches={res['n_val']}  bicubic PSNR={bicubic:.3f} dB")
table

In [ ]:
import matplotlib.pyplot as plt

# Color follows the config, never its position in the run (validated categorical slots).
COLORS = {"K2": "#2a78d6", "K4": "#eb6834"}
INK, MUTED, GRID = "#1f1f1e", "#6b6a63", "#e4e3dd"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, key, title, unit in [(axes[0], "psnr", "Validation PSNR", "dB"),
                             (axes[1], "ssim", "Validation SSIM", "")]:
    for name in NAMES:
        h = R[name]["history"]
        x = [p["iter"] for p in h]
        y = [p[key] for p in h]
        ax.plot(x, y, color=COLORS.get(name), linewidth=2, marker="o", markersize=4, label=name)
    if key == "psnr":
        ax.axhline(bicubic, color=MUTED, linewidth=1.2, linestyle="--")
        ax.annotate(f"bicubic {bicubic:.2f}", (0, bicubic), xytext=(4, 4), textcoords="offset points",
                    xycoords=("axes fraction", "data"), color=MUTED, fontsize=9)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel("training iteration", color=MUTED)
    ax.set_ylabel(unit, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.tick_params(colors=MUTED)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(OUT_DIR / "curves.png", dpi=150)
plt.show()

In [ ]:
# Every check, as a table (same numbers as the plot).
pd.DataFrame({
    (name, key): {p["iter"]: round(p[key], 4) for p in R[name]["history"]}
    for name in NAMES for key in ("psnr", "ssim")
})

## Conclusion

*Fill in after the run.*

* **Quality:** which config has the higher PSNR / SSIM at most checks (not just one), and by how much.
* **vs bicubic:** does the better config beat the bicubic baseline?
* **Cost:** GFLOPs and forward time saved by K4 over K2 (the parameter count is the same).
* **Recommendation:** K2 or K4, and whether a longer run or a second seed (`SEED = 1`) is needed to confirm it.

**Limits of this experiment:** one seed, batch size 1, a short fixed budget (not trained to
convergence), validation on 32 patches from the train split. The test split is untouched and is
for the final model evaluation (`inference_colab.ipynb`).